# Ch 18. GPT Anatomy — Solutions

> This notebook provides reproducible solutions to all five exercises. Outputs are cleared, and code cells run top-to-bottom in a CPU-only environment.


## Problem 1

**Problem**: Scale MiniGPT to $d_{model}=64,128,256$ and compare parameter counts and forward-pass time.

### Solution

At fixed depth and vocabulary, block parameters scale mainly as $O(Ld^2)$ and embeddings as $O(Vd)$. Timing requires device warm-up, synchronization, and a median over repetitions. The code verifies deterministic parameter scaling and does not fabricate timings.


In [ ]:
import time
import numpy as np

rng = np.random.default_rng(1801)
batch, repeats = 64, 30
results = {}
for width in (64, 128, 256):
    X = rng.normal(size=(batch, width)); W = rng.normal(size=(width, width))
    for _ in range(3): X @ W
    samples = []
    for _ in range(repeats):
        start = time.perf_counter(); X @ W; samples.append(time.perf_counter() - start)
    results[width] = {"block_parameters": 12 * width**2, "median_ms": 1000 * float(np.median(samples))}
assert [results[d]["block_parameters"] for d in results] == [12*d*d for d in results]
print({d: {"parameters": r["block_parameters"], "median_ms": round(r["median_ms"], 4)} for d, r in results.items()})


## Problem 2

**Problem**: Disable weight tying and compare parameter count and performance.

### Solution

Weight tying shares input embedding $E$ with output-logit matrix $E^T$. Disabling it adds exactly $Vd$ parameters. Performance is data-dependent, so report validation loss under equal token budgets and seeds.


In [ ]:
import numpy as np

rng = np.random.default_rng(1802)
vocab, width, samples = 40, 12, 400
tokens = rng.integers(0, vocab, size=samples)
embedding = rng.normal(size=(vocab, width))
features = embedding[tokens]
labels = tokens
untied = rng.normal(scale=0.1, size=(width, vocab))
lr = 0.8
losses = []
for _ in range(80):
    logits = features @ untied; logits -= logits.max(1, keepdims=True)
    probs = np.exp(logits); probs /= probs.sum(1, keepdims=True)
    losses.append(float(-np.log(probs[np.arange(samples), labels]).mean()))
    probs[np.arange(samples), labels] -= 1
    untied -= lr * features.T @ probs / samples
tied_logits = features @ embedding.T
tied_accuracy = float(np.mean(tied_logits.argmax(1) == labels))
assert losses[-1] < losses[0] and tied_accuracy > 0.9
print({"extra_untied_parameters": untied.size, "untied_loss_before_after": [round(losses[0],3), round(losses[-1],3)], "tied_accuracy": tied_accuracy})


## Problem 3

**Problem**: Change model depth to $L=2,4,8$ and measure forward-pass time.

### Solution

At fixed $d,T$, forward computation is approximately linear in depth $L$. Actual latency need not be exactly linear because of runtime overhead, so use warmed-up repetitions and report uncertainty.


In [ ]:
import time
import numpy as np

rng = np.random.default_rng(1803)
X = rng.normal(size=(96, 96)); W = rng.normal(scale=0.05, size=(96, 96))
results = {}
for layers in (2, 4, 8):
    samples = []
    for _ in range(12):
        h = X.copy(); start = time.perf_counter()
        for _ in range(layers): h = np.tanh(h @ W)
        samples.append(time.perf_counter() - start)
    results[layers] = 1000 * float(np.median(samples))
assert all(value > 0 for value in results.values())
print({layers: {"median_forward_ms": round(ms, 4), "time_per_layer_ms": round(ms/layers, 4)} for layers, ms in results.items()})


## Problem 4

**Problem**: Remove embedding scaling by $\sqrt{d}$ and observe training stability.

### Solution

If embedding-component variance is fixed, multiplying by $\sqrt d$ increases the initial residual-stream scale. The effect of removal depends on LN placement and initialization; assess finite loss, gradient norms, and divergence rate over seeds.


In [ ]:
import numpy as np

rng = np.random.default_rng(1804)
width, batch = 128, 256
embedding = rng.normal(scale=1 / np.sqrt(width), size=(batch, width))
target = rng.normal(size=(batch, width))
results = {}
for scale_name, scale in (("none", 1.0), ("sqrt_d", np.sqrt(width))):
    h = embedding * scale
    loss = float(np.mean((h-target)**2))
    gradient_norm = float(np.linalg.norm(2*(h-target)*scale / h.size))
    results[scale_name] = {"stream_std": float(h.std()), "loss": loss, "gradient_norm": gradient_norm}
assert results["sqrt_d"]["stream_std"] > 8 * results["none"]["stream_std"]
print({k: {m: round(v, 5) for m, v in values.items()} for k, values in results.items()})


## Problem 5

**Problem**: This model will be trained in Ch 19. Design how the pretraining data should be prepared.

### Solution

A reproducible pipeline performs license checks, normalization and deduplication, document-level train/validation split, fixed tokenization, EOS joining, then blocking into length $T+1$. Deduplicate before splitting to prevent leakage, and record data version, hashes, and token counts.


In [ ]:
import numpy as np

documents = ["attention uses context", "context guides prediction", "tokens form sequences", "sequences train models"]
normalized = [" ".join(doc.lower().split()) for doc in documents]
unique = list(dict.fromkeys(normalized))
split = int(0.75 * len(unique)); train_docs, validation_docs = unique[:split], unique[split:]
vocabulary = {word: i + 1 for i, word in enumerate(sorted(set(" ".join(train_docs).split())))}
eos = 0
train_tokens = [vocabulary[word] for doc in train_docs for word in doc.split()] + [eos]
block_length = 4
blocks = [train_tokens[i:i+block_length+1] for i in range(0, len(train_tokens)-block_length, block_length)]
assert set(train_docs).isdisjoint(validation_docs) and all(len(block) == block_length + 1 for block in blocks)
print({"documents": len(documents), "unique": len(unique), "train_validation": [len(train_docs), len(validation_docs)],
       "vocabulary_size": len(vocabulary)+1, "training_blocks": len(blocks)})
